# 第十一週：Text Embeddings

本週課程為「Text Embeddings」，主要會使用到gensim套件來實現自行訓練word2vec模型以及使用預訓練word2vec模型，以及透過sentence-transformers套件和API取得embeddings。

### 大綱：
1.   DEMO
*   Word2Vec
  *   自己訓練w2v模型
  *   使用別人訓練好的w2v模型
*   Transformers Embeddings
  *   小模型（BERT）：
      *   不同語言的BERT：uncased / chinese / multilingual
  *   大模型（LLM）：
      *   API based
      *   Open Source LLM  

2.   資料集實作任務
*    使用embedding 做 NLP 任務
  *   找相似文件（文章）
  *   文件分類任務


### 套件安裝

這邊套件要自己裝

In [ ]:
!pip install jieba
!pip install "gensim==4.3.3" "spacy==3.7.2" "thinc==8.2.2"

如果安裝套件時遇到跟 numpy 相關錯誤的話，可以嘗試執行以下程式碼

In [ ]:
# # 先清除舊的 numpy
# !pip uninstall -y numpy

# # 安裝對應版本
# !pip install numpy==1.26.4 gensim==4.3.3 spacy==3.7.2 thinc==8.2.2

# # 強制重啟 Runtime
# import os
# os.kill(os.getpid(), 9)

In [ ]:
!pip install plotly

In [ ]:
import pandas as pd
import jieba
import jieba.analyse
import re
import numpy as np
from collections import defaultdict
import multiprocessing

from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec, KeyedVectors


import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import seaborn as sns
import torch

from matplotlib.font_manager import fontManager
import plotly.express as px

sns.set_style("darkgrid")

### 資料前處理

讀入吃到飽示範資料集

In [ ]:
# 設定繁體中文詞庫
jieba.set_dictionary('dict\dict.txt')
jieba.load_userdict('./dict/user_dict.txt')

# 新增stopwords
with open('./dict/stopwords.txt',encoding="utf-8") as f:
    stopWords = [line.strip() for line in f.readlines()]

In [ ]:
# 斷詞函式
def getToken(row):
    seg_list = jieba.lcut(row)
    seg_list = [w for w in seg_list if w not in stopWords and len(w)>1] # 篩選掉停用字與字元數小於1的詞彙

    return seg_list

In [ ]:
# 讀入中文示範資料集
origin_data = pd.read_csv('raw_data/starlux_dataset.csv')

In [ ]:
# 資料前處理

# 去除一些不需要的欄位
# metaData = origin_data.drop(['artPoster', 'artCatagory', 'artComment', 'e_ip', 'insertedDate', 'dataSource'], axis=1)
# 修改後的寫法
metaData = origin_data.drop(
    ['artPoster', 'artCatagory', 'artComment', 'e_ip', 'insertedDate', 'dataSource'], 
    axis=1, 
    errors='ignore'
)
# 只留下中文字
metaData['sentence'] = metaData['artContent'].str.replace(r'\n\n','。', regex=True)
metaData['sentence'] = metaData['sentence'].str.replace(r'\n','', regex=True)

metaData['sentence'] = metaData['sentence'].str.split("[,，。！!？?]{1,}")
metaData = metaData.explode('sentence').reset_index(drop=True)

metaData['sentence'] = metaData['sentence'].apply(lambda x: re.sub('[^\u4e00-\u9fff]+', '',x))

metaData['word'] = metaData.sentence.apply(getToken)

metaData = metaData[metaData['word'].apply(len) > 0]

metaData.head(10)

### word2vec


####（1）如何自己訓練word2vec模型

建立訓練資料時將考慮bigram，以下為Phrases函式的使用示範

In [ ]:
docs = ['new york is united states', 'new york is most populated city in the world','i love to stay in new york']

token_ = [doc.split(" ") for doc in docs]
# Phrases 建立bigram,
# 少於min_count的字的字或bigrams會被忽略,
# 大於threshold的bigrams會被加入
bigram = Phrases(token_, min_count=1, threshold=2)
bigram_phraser = Phraser(bigram)

for sent in token_:
    print(sent)  #處理前
    print("=> ",bigram_phraser[sent]) # 處理後，還是原本的句子，只是有抓出的bigram片語會被合併

實際應用在資料集上

In [ ]:
sents = metaData['word'].to_list()
bigrams = Phrases(sents,min_count=1, threshold=1000)
bigram_phrasers = Phraser(bigrams)
metaData['word_list_bigrams'] = list(bigram_phrasers[sents])

metaData.head()

In [ ]:
word_freq = defaultdict(int)
# 計算詞頻
sents = metaData['word_list_bigrams'].tolist()
for sent in sents: # sent 中的每個句子
    for i in sent: # i 是句子中的每個字
        word_freq[i] += 1

In [ ]:
print(f"total unique words in sentences: {len(word_freq)}")
sorted(word_freq, key=word_freq.get, reverse=True)[:10]

In [ ]:
print(f"sentence number of corpus: {len(sents)}")
i = 0
for sent in sents:
    i = i + len(sent)
print(f"average length of sentences: {i/len(sents)}")

Word2Vec 計算

In [ ]:
# 環境變數設定
%env PYTHONHASHSEED=2025

In [ ]:
# 查看機器的core
cores = multiprocessing.cpu_count()
print(f"number of cores: {cores}")

In [ ]:
# 建立模型
w2v_model = Word2Vec(sents,
                     min_count=20,# 小於20次tf的字會被刪除
                     window=2,# 往左右各2的距離
                     vector_size=128,# vector 的維度
                     sample=0.005,# 愈小的話，高tf的字會不容易被選到
                     alpha=0.001,# learning rate
                     min_alpha=0.0005, # 迭代到最小的learning rate，learning rate會慢慢下降至min_alpha
                     negative=0,
                     workers=cores-1, # 用的cpu資源
                     seed=8787,
                     sg = 1,# 0/1 是否使用skip gram
                     epochs= 30,
                     hs=1 , # hierarchical softmax
                     )

查看結果

In [ ]:
# 檢查最相關的字
w2v_model.wv.most_similar('飛機',topn=10)

In [ ]:
w2v_model.wv.most_similar('年終',topn=10)

In [ ]:
w2v_model.wv.most_similar(['員工','年終'],topn=10)

In [ ]:
# 跟兩個字最不相關
w2v_model.wv.most_similar(negative=['年終','飛機'],topn=10)

In [ ]:
# 計算兩個字之間的關係
w2v_model.wv.similarity("年終","長榮")

In [ ]:
w2v_model.wv.similarity("星宇","飛機")

In [ ]:
# 比較字詞間，誰最不相關（邊緣）
w2v_model.wv.doesnt_match(["年終", "星宇", '飛機'])

In [ ]:
# 相對關係
w2v_model.wv.most_similar(positive=["年終"], negative=["星宇"], topn=5)

In [ ]:
# 取得所有的字
words = w2v_model.wv.key_to_index.keys()

視覺化字之間的關係及將字做分群

In [ ]:
# 降維：利用PCA tSNE

def reduceDim(mat,method:str='PCA',dim:str=2,perplexity = 25,learning_rate = 400):

    method_dict = {
        "PCA":PCA(n_components=dim,iterated_power = 1000,random_state=0),
        "TSNE":TSNE(n_components=dim,random_state=0,perplexity=perplexity,learning_rate=learning_rate),
    }
    new_feat = method_dict[method].fit_transform(mat)

    return new_feat


In [ ]:
# 拿到list of words 的vector
def getVecs(model,words:list):
    vecs = []
    for i in words:
        vecs.append(model.wv[i])
    return np.vstack(vecs)


In [ ]:
getVecs(w2v_model,['年終','星宇'])

In [ ]:
# 擴展相似的字詞
def expandPosWord(model, words:list, top_n:int, split = True):

    if split == False:
        wp = model.wv.most_similar(words,topn = top_n)
        return wp
    expand = []

    for w in words:
        wp = model.wv.most_similar(w,topn = top_n)
        for i in wp:
            expand.append(i[0])

    return list(set(expand))


In [ ]:
expandPosWord(w2v_model,['年終','飛機'],top_n = 10)

In [ ]:
# 畫出兩維的散佈圖
def plotScatter(vec_df):
    """
    vec_df: 字詞及其兩個維度的值
    """
    plt.figure(figsize=(15,15))
    fontManager.addfont('./TaipeiSansTCBeta-Regular.ttf')
    plt.rcParams['font.sans-serif'] = ['Taipei Sans TC Beta']
    plt.rcParams['font.size'] = '16'

    p = sns.scatterplot(x="dim1", y="dim2",
                  data=vec_df)
    for line in range(0, vec_df.shape[0]):
         p.text(vec_df["dim1"][line],
                 vec_df['dim2'][line],
                 '  ' + vec_df["word"][line].title(),
                 horizontalalignment='left',
                 verticalalignment='bottom', size='medium',
                 weight='normal'
                ).set_size(15)
    plt.show()

# 畫出三維的散佈圖
def plotScatter3D(vec_df):
    vec_df['size'] = .5
    if 'color' not in vec_df.columns:
        vec_df['color'] = 'blue'
    fig = px.scatter_3d(
        vec_df,'dim1','dim2','dim3',text = 'word',width=800, height=800,color = 'color',size = 'size'

    )

    fig.show()

In [ ]:
# 查看總數
total_words = len(words)
print(f"目前詞庫總數：{total_words}")

# 查看前 20 個詞（確認過濾邏輯是否正確）
print("詞庫樣本：", list(words)[:20])

In [ ]:
sample_words = np.random.choice(list(words),300,replace=False).tolist()

feat = getVecs(model=w2v_model,words=sample_words)
print(feat.shape)
new_feat = reduceDim(feat,method='TSNE',perplexity=20,learning_rate = 300)
print(new_feat.shape)

In [ ]:
word_df = pd.DataFrame({
    "word":sample_words,
    "dim1":new_feat[:,0],
    "dim2":new_feat[:,1],
})

In [ ]:
plotScatter(word_df)

3D 散狀圖

In [ ]:
new_feat = reduceDim(feat,dim = 3,method = 'PCA' )
print(new_feat.shape)
word_df = pd.DataFrame({
    "word":sample_words,
    "dim1":new_feat[:,0],
    "dim2":new_feat[:,1],
    "dim3":new_feat[:,2],
})
plotScatter3D(word_df)

將字分群

In [ ]:
!pip install scikit-learn-extra

In [ ]:
# kmeans分群
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
# 只使用word vector 去分群
def cluster(X,method = 'kmeans',n = 2):

    method_dict = {
        'kmeans':KMeans(n_clusters=n, random_state=0),
        'kmedos':KMedoids(n_clusters=n, random_state=0)
    }
    method_dict[method].fit(X)
    result = method_dict[method].predict(X)
    return result


In [ ]:
new_feat = reduceDim(feat,method='PCA',dim = 20)
d3_feat = reduceDim(feat,method='PCA',dim = 3)
word_df = pd.DataFrame({
    "word":sample_words,
    "color":cluster(new_feat,n=4),
    "dim1":d3_feat[:,0],
    "dim2":d3_feat[:,1],
    "dim3":d3_feat[:,2],

})
plotScatter3D(word_df)

### Transformers Embeddings

#### 使用 Sentence-Transformer 套件   
參考資料：https://www.sbert.net/index.html

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, models, util

#### 小模型，以BERT為範例

因為後續在查看結果時會一直使用到此程式碼，所以包成function以便之後使用。

In [ ]:

def get_result_df(sentences, cosine_scores):

  result = []
  for i in range(len(sentences)):
      for j in range(i+1, len(sentences)):
          result.append([sentences[i], sentences[j], cosine_scores[i][j].item()])

  result_df = pd.DataFrame(result, columns=["sentence1", "sentence2", "score"])
  result_df = result_df.sort_values("score", ascending = False)

  return result_df

接下來將以針對中文 BERT 模型做範例。    
中文 bert-base-chinese

In [ ]:
# 中文 bert-base-chinese
bert_ch = SentenceTransformer('google-bert/bert-base-chinese')

bert_ch.tokenizer.add_special_tokens({'pad_token': '[PAD]'})


# 範例句子 
sentences = [
    "星宇航空的服務真的很棒",
    "這家航空公司的飛機餐很好吃",
    "我不喜歡這次的飛行體驗"
]

# 使用 encode() 對資料做embedding
embeddings_ch = bert_ch.encode(sentences)

# Compute cosine-similarities
cosine_scores = util.cos_sim(embeddings_ch, embeddings_ch)

# 印出句子間的cosine similarity分數
result_df = get_result_df(sentences, cosine_scores)
result_df

針對以上使用 `bert-base-chinese` 產出的相似度分數普遍偏高（約 **0.78 - 0.83**）且區別度不足的現象，分析如下：

1. 為什麼分數集中在 0.78 ~ 0.83 之間？

* 字面重疊度高 ：
    這三個句子共享了強烈的「航空語境」（例如：*航空公司、飛行、服務、體驗*）。對於原生的 BERT 模型而言，它會捕捉到這些詞彙的高度共現性，因此在向量空間中認為這些句子的主題極其接近。
* 缺乏對比訓練 ：
    `bert-base-chinese` 最初的訓練目標是「遮罩語言模型 (MLM)」與「下一句預測 (NSP)」，即預測填空。它並非專門為了區分「正面」與「負面」情感而設計。在模型看來，「服務很棒」和「不喜歡體驗」都屬於「描述飛行感受」的範疇，因此向量距離非常接近。
* 模型侷限：非等向性 ：
    原生 BERT 產出的向量空間通常會呈現錐形分佈，導致所有句子的向量都擠在一個狹小的區域。這種「非等向性」問題會使得即使語義完全相反的句子，其餘弦相似度分數依然維持在高位（如 0.78 以上）。


 2. 數值代表的具體含義

* **0.83 (服務很棒 vs 飛機餐很好吃)**：
    這兩句皆為「正面評價」，且主題皆與航空服務高度相關，因此在空間中距離最近，分數最高。
* **0.78 (正面讚美 vs 負面體驗)**：
    雖然語義上一個是讚美、一個是不喜歡，但由於兩者討論的客體都是「航空體驗」，在基礎模型未經細節微調的情況下，仍會給予極高的相似度評價。


上面課堂中用的模型 結果出來「正評」跟「負評」的分數會太過接近 (0.78-0.83) 故換成以下這個模型  

In [ ]:
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
bert_ch = SentenceTransformer(model_name)

# 範例句子 
sentences = [
    "星宇航空的服務真的很棒",
    "這家航空公司的飛機餐很好吃",
    "我不喜歡這次的飛行體驗"
]

# 使用 encode() 對資料做embedding
embeddings_ch = bert_ch.encode(sentences)

# Compute cosine-similarities
cosine_scores = util.cos_sim(embeddings_ch, embeddings_ch)

# 印出句子間的cosine similarity分數
result_df = get_result_df(sentences, cosine_scores)
result_df

以上語義相似度分數結果分析

1. 分數高 (0.54) = 語境與情緒接近
* **分析**：「服務棒」與「飛機餐好吃」雖然描述的對象不同，但兩者皆為**正面評價**且都圍繞著**航空體驗**。
* **意義**：模型認為這兩句話在向量空間中的「方向」比較一致，具備較強的關聯性。

2. 分數中 (0.41) = 主題相關但情緒不同
* **分析**：「飛機餐好吃」與「不喜歡飛行體驗」雖然主題都是航空，但存在一正一負的情緒差異。
* **意義**：模型察覺到情緒上的拉伸（情緒位移），因此分數明顯比純粹正面的組合還要低。

3. 分數低 (0.30) = 語義衝突最明顯
* **分析**：「服務真的很棒」與「我不喜歡這次體驗」在語義上是完全相反的（極好 vs 不滿）。
* **意義**：這兩句話在向量空間中的距離最遠，代表模型成功識別出它們是語義衝突的對立面。    

### 使用embedding做NLP任務

#### 相似文件

In [ ]:
df_similar = origin_data[['system_id','artTitle', 'artContent']]
df_similar['artContent'] = df_similar['artContent'].apply(lambda x: re.sub('[^\u4e00-\u9fff]+', '',x))

df_similar.head(5)

使用 bert-base-chinese 做示範

In [ ]:
# 中文 bert-base-chinese
bert_ch = SentenceTransformer('google-bert/bert-base-chinese', device='cuda')

bert_ch.tokenizer.add_special_tokens({'pad_token': '[PAD]'})

取得整個文集的 embeddings

In [ ]:
corpus_embeddings = bert_ch.encode(
    df_similar['artContent'],
    convert_to_tensor=True,
    batch_size=32
)

In [ ]:
query_num = 6 # 指定文章

# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = 5


query_embedding = bert_ch.encode(df_similar['artContent'][query_num], convert_to_tensor=True)

# We use cosine-similarity and torch.topk to find the highest 5 scores
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
top_results = torch.topk(cos_scores, k=top_k)

print("\n\n======================\n\n")
print("Query:", df_similar['artTitle'][query_num])
print("\n 資料集中前五相似的文章:")

for score, idx in zip(top_results[0], top_results[1]):
    print(df_similar['artTitle'][idx.item()], "(Score: {:.4f})".format(score))

print("\n\n======================\n\n")

In [ ]:
query_num = 30

top_k = 5

query_embedding = bert_ch.encode(df_similar['artContent'][query_num], convert_to_tensor=True)

# We use cosine-similarity and torch.topk to find the highest 5 scores
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
top_results = torch.topk(cos_scores, k=top_k)

print("\n\n======================\n\n")
print("Query:", df_similar['artTitle'][query_num])
print("\n 資料集中前五相似的文章:")

for score, idx in zip(top_results[0], top_results[1]):
    print(df_similar['artTitle'][idx.item()], "(Score: {:.4f})".format(score))

print("\n\n======================\n\n")

### 分類任務
使用bert-base-chinese模型對新聞資料集做embeddings，接著訓練分類器。（參考week7程式碼）


In [ ]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
# !pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, models, util

In [ ]:
# 中文 bert-base-chinese
bert_ch = SentenceTransformer('google-bert/bert-base-chinese', device='cuda')

bert_ch.tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [ ]:
udn = pd.read_csv("raw_data/starlux_dataset.csv")
udn.head(3)

In [ ]:
# 過濾 nan 的資料
udn = udn.dropna(subset=['artTitle'])
udn = udn.dropna(subset=['artContent'])
# 移除網址格式
udn["artContent"] = udn.artContent.apply(
    lambda x: re.sub("(http|https)://.*", "", x)
)
udn["artTitle"] = udn["artTitle"].apply(
    lambda x: re.sub("(http|https)://.*", "", x)
)
# 只留下中文字
udn["artContent"] = udn.artContent.apply(
    lambda x: re.sub("[^\u4e00-\u9fa5]+", "", x)
)
udn["artTitle"] = udn["artTitle"].apply(
    lambda x: re.sub("[^\u4e00-\u9fa5]+", "", x)
)

# 留下 content
udn["content"] = udn["artTitle"] + udn["artContent"]
udn = udn[["content", "artUrl", "artCatagory"]]  # 文章內容 文章連結
udn.head()

udn["embeddings"] = udn.content.apply(lambda x: bert_ch.encode(x))
udn.head(3)

In [ ]:
import numpy as np
from ast import literal_eval

In [ ]:
data = udn.copy()

X = data["embeddings"].apply(pd.Series)
y = data["artCatagory"]

# 把整個資料集七三切
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=777
)

print(X_train.head())
print(y_train.head())

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)
clf

In [ ]:
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)
print(y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
## Accuracy, Precision, Recall, F1-score
print(classification_report(y_test, y_pred))

---
## 資料集實作任務 — 使用 Embedding 做 NLP 任務

### 任務一：找相似文件（文章）

利用 `paraphrase-multilingual-MiniLM-L12-v2` 多語言 Sentence-BERT 取得每篇文章的 embedding，以餘弦相似度找出最相關的文章。

In [ ]:
import re
import torch
from sentence_transformers import SentenceTransformer, util

# 讀入資料並前處理
df_sim = pd.read_csv('raw_data/starlux_dataset.csv')
df_sim = df_sim.dropna(subset=['artTitle', 'artContent']).reset_index(drop=True)

def clean_text(text):
    return re.sub('[^一-鿿]+', '', str(text))

df_sim['text_clean'] = (
    df_sim['artTitle'].apply(clean_text) + df_sim['artContent'].apply(clean_text)
)
df_sim = df_sim[df_sim['text_clean'].str.len() > 10].reset_index(drop=True)

print(f"共 {len(df_sim)} 篇文章，類別分布：")
print(df_sim['board'].value_counts())

# 載入多語言 Sentence-BERT，對全部文章做 embedding
sim_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("\n正在對所有文章做 embedding，請稍候...")
corpus_embeddings = sim_model.encode(
    df_sim['text_clean'].tolist(),
    convert_to_tensor=True,
    batch_size=32,
    show_progress_bar=True
)
print(f"Embedding shape: {corpus_embeddings.shape}")

In [ ]:
def find_similar_articles(query_idx, top_k=5):
    """輸入文章 index，顯示前 top_k 篇最相似文章"""
    query_emb = corpus_embeddings[query_idx]
    cos_scores = util.cos_sim(query_emb, corpus_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k + 1)  # +1 排除自身

    title_q = df_sim['artTitle'].iloc[query_idx]
    board_q = df_sim['board'].iloc[query_idx]
    print(f"Query（index={query_idx}）[{board_q}]：【{title_q}】")
    print("=" * 70)

    count = 0
    for score, idx in zip(top_results[0], top_results[1]):
        if idx.item() == query_idx:
            continue
        title = df_sim['artTitle'].iloc[idx.item()]
        board = df_sim['board'].iloc[idx.item()]
        print(f"  Top {count+1}. [{board}]【{title}】  相似度：{score.item():.4f}")
        count += 1
        if count >= top_k:
            break
    print()

# 以三篇不同的文章示範
for qidx in [0, 50, 100]:
    find_similar_articles(qidx, top_k=5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import fontManager

fontManager.addfont('./TaipeiSansTCBeta-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Taipei Sans TC Beta']

# 取前 20 篇文章，計算兩兩相似度並畫 heatmap
sample_idx = list(range(20))
sample_labels = [
    f"[{df_sim['board'].iloc[i][:3]}]{df_sim['artTitle'].iloc[i][:8]}..."
    for i in sample_idx
]
sample_embs = corpus_embeddings[sample_idx]
sim_matrix = util.cos_sim(sample_embs, sample_embs).cpu().numpy()

plt.figure(figsize=(12, 10))
sns.heatmap(
    sim_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=sample_labels, yticklabels=sample_labels,
    annot_kws={'size': 7}
)
plt.title('文章間餘弦相似度矩陣（前 20 篇）', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

### 任務二：文件分類任務

以文章 embedding 作為特徵，訓練 **Logistic Regression** 分類器，預測文章屬於 **Aviation**（航空板）還是 **Stock**（股票板）。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 用 embedding 作為特徵，board 作為標籤
X = corpus_embeddings.cpu().numpy()
y = df_sim['board']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"訓練集: {len(X_train)} 筆  |  測試集: {len(X_test)} 筆\n")

# 訓練 Logistic Regression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("=== Logistic Regression 分類結果 ===\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

In [ ]:
# 混淆矩陣視覺化
classes = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=classes)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=classes, yticklabels=classes
)
plt.title('混淆矩陣（Logistic Regression）')
plt.ylabel('真實標籤')
plt.xlabel('預測標籤')
plt.tight_layout()
plt.show()